# IFRS S1/S2 report generation — clean LangGraph engine (on your Azure client)

From-scratch, layered rewrite of the section-generation loop. It **reuses your notebook's own
config and Azure client** (the two setup cells below, copied verbatim) and **loads your prep
artifacts** from `DIRS`. Run your prep phase once to produce the per-section
`evidence_map_*`/`coverage_matrix_*`/`disclosure_plan_*` files; this engine does the generation.

Layers: settings → typed models → io → llm (`AzureChatModel` wrapping your `azure_chat`) →
tools → deterministic gates → structured judges → deterministic approval → tool-using agents →
LangGraph loop → assembly. Only dependency: `pip install langgraph`.

## Your setup — config (verbatim from the old notebook)

In [ ]:
# CELL 1 — SETUP PATHS AND CONFIG

import os
from pydoc import resolve
import re
import json
import time
import uuid
import shutil
import random
import urllib.request
import urllib.error
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict, Counter
import pandas as pd
try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    raise ImportError('Install python-dotenv first: pip install python-dotenv')
try:
    load_dotenv(find_dotenv(usecwd=True), override=True)
except TypeError:
    load_dotenv(find_dotenv(), override=True)
except AssertionError:
    load_dotenv(Path.cwd() / '.env', override=True)
CURRENT_DIR = Path.cwd().resolve()
if CURRENT_DIR.name == 'notebooks':
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks').exists():
    NOTEBOOK_DIR = (CURRENT_DIR / 'notebooks').resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR
GEN_DATA_DIR = NOTEBOOK_DIR / 'gen_data'
PAYLOAD_DIR = Path(os.getenv('PAYLOAD_DIR', GEN_DATA_DIR / 'payloads')).resolve()
REQUIREMENTS_DIR = Path(os.getenv('IFRS_REQUIREMENTS_DIR', GEN_DATA_DIR / 'IFRS' / 'ifrs_requirements_kb_outputs_final' / 'section_by_section_requirements' / 'json')).resolve()
STYLE_SYSTEM_DIR = Path(os.getenv('STYLE_SYSTEM_DIR', GEN_DATA_DIR / 'style' / 'style_system')).resolve()
OUTPUT_DIR = Path(os.getenv('GENERATION_OUTPUT_DIR', GEN_DATA_DIR / 'generated_reports' / 'agentic_ifrs_report')).resolve()
PIPELINE_MODE = os.getenv('PIPELINE_MODE', 'synthetic_demo')
FORBID_INVENTION = True
ALLOW_PARTIAL_COVERAGE = os.getenv('ALLOW_PARTIAL_COVERAGE', 'true').lower() == 'true'
USE_FUZZY_EVIDENCE_MAPPER = os.getenv('USE_FUZZY_EVIDENCE_MAPPER', 'false').lower() == 'true'
MAX_REVISION_LOOPS = int(os.getenv('MAX_REVISION_LOOPS', '2'))
SECTIONS = ['General Requirements', 'Governance', 'Strategy', 'Risk Management', 'Metrics and Targets']
SECTION_SLUGS = {'General Requirements': 'general_requirements', 'Governance': 'governance', 'Strategy': 'strategy', 'Risk Management': 'risk_management', 'Metrics and Targets': 'metrics_and_targets'}
DIRS = {'evidence_maps': OUTPUT_DIR / '01_evidence_maps', 'coverage': OUTPUT_DIR / '02_coverage', 'missing_requirements': OUTPUT_DIR / '03_missing_requirements', 'plans': OUTPUT_DIR / '04_disclosure_plans', 'drafts': OUTPUT_DIR / '05_draft_sections', 'claims': OUTPUT_DIR / '06_claims_registers', 'gates': OUTPUT_DIR / '07_deterministic_gates', 'judges': OUTPUT_DIR / '08_judge_results', 'revisions': OUTPUT_DIR / '09_revised_sections', 'approved': OUTPUT_DIR / '10_approved_sections', 'connectivity': OUTPUT_DIR / '11_connectivity', 'handoff': OUTPUT_DIR / '12_pdf_handoff', 'audit_logs': OUTPUT_DIR / 'audit_logs'}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)
print('Current working directory:', CURRENT_DIR)
print('Notebook directory:', NOTEBOOK_DIR)
print('Payload directory:', PAYLOAD_DIR)
print('Requirements directory:', REQUIREMENTS_DIR)
print('Style system directory:', STYLE_SYSTEM_DIR)
print('Output directory:', OUTPUT_DIR)
print('Pipeline mode:', PIPELINE_MODE)
print('Forbid invention:', FORBID_INVENTION)
print('Use fuzzy mapper:', USE_FUZZY_EVIDENCE_MAPPER)

## Your setup — Azure client (verbatim from the old notebook)

In [ ]:
# CELL 2 — LLM CLIENT

import http.client
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY')
AZURE_OPENAI_GPT52_DEPLOYMENT_URL = os.getenv('AZURE_OPENAI_GPT52_DEPLOYMENT_URL')
AZURE_OPENAI_FAST_DEPLOYMENT_URL = os.getenv('AZURE_OPENAI_FAST_DEPLOYMENT_URL')

def _clean_url(value: Optional[str]) -> Optional[str]:
    """
    Basic cleanup for full deployment URLs.

    Keeps the full URL as provided; does not add/replace api-version.
    Handles:
    - surrounding quotes
    - accidental markdown link format: [label](https://...)
    - accidental copied bracket+url format
    """
    if not value:
        return None
    value = str(value).strip().strip('"').strip("'").strip()
    md_match = re.search('\\]\\((https://[^)\\s]+)\\)', value)
    if md_match:
        value = md_match.group(1).strip()
    https_positions = [m.start() for m in re.finditer('https://', value)]
    if https_positions:
        value = value[https_positions[-1]:]
    value = value.strip().strip('[]').strip()
    value = value.rstrip(').,;')
    return value
AZURE_OPENAI_GPT52_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_GPT52_DEPLOYMENT_URL)
AZURE_OPENAI_FAST_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_FAST_DEPLOYMENT_URL)
if not AZURE_OPENAI_FAST_DEPLOYMENT_URL:
    AZURE_OPENAI_FAST_DEPLOYMENT_URL = AZURE_OPENAI_GPT52_DEPLOYMENT_URL
MODEL_CONFIG = {'fuzzy_evidence_mapper': 'fast', 'section_writer': 'strong', 'claims_register_builder': 'strong', 'ifrs_coverage_judge': 'strong', 'evidence_judge': 'strong', 'style_judge': 'fast', 'minimal_reviser': 'strong', 'whole_report_connectivity_judge': 'strong'}

def _mask_url_for_display(url: Optional[str]) -> str:
    """Mask full endpoint URL while keeping enough shape for diagnostics."""
    if not url:
        return 'NOT CONFIGURED'
    try:
        import urllib.parse
        parsed = urllib.parse.urlparse(url)
        host = parsed.netloc
        if host:
            host_parts = host.split('.')
            if host_parts and len(host_parts[0]) > 6:
                host_parts[0] = host_parts[0][:3] + '***' + host_parts[0][-2:]
            host = '.'.join(host_parts)
        path = parsed.path
        path = re.sub('(/deployments/)([^/]+)(/chat/completions)', lambda m: m.group(1) + m.group(2)[:2] + '***' + m.group(3), path)
        query = '...' if parsed.query else ''
        return urllib.parse.urlunparse((parsed.scheme, host, path, '', query, ''))
    except Exception:
        return '<configured URL, masking failed>'

def validate_llm_config() -> None:
    required = {'AZURE_OPENAI_API_KEY': AZURE_OPENAI_API_KEY, 'AZURE_OPENAI_GPT52_DEPLOYMENT_URL': AZURE_OPENAI_GPT52_DEPLOYMENT_URL, 'AZURE_OPENAI_FAST_DEPLOYMENT_URL': AZURE_OPENAI_FAST_DEPLOYMENT_URL}
    missing = [name for name, value in required.items() if not value]
    if missing:
        flags = {'api_key_loaded': bool(AZURE_OPENAI_API_KEY), 'gpt52_url_loaded': bool(AZURE_OPENAI_GPT52_DEPLOYMENT_URL), 'fast_url_loaded': bool(AZURE_OPENAI_FAST_DEPLOYMENT_URL)}
        raise ValueError('Missing Azure/OpenAI full-URL configuration values: ' + ', '.join(missing) + '\n\nLoaded flags, keys are never printed:\n' + json.dumps(flags, indent=2) + '\n\nRequired .env:\nAZURE_OPENAI_API_KEY=<shared Azure resource key>\nAZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full GPT-5.2 deployment URL>\nAZURE_OPENAI_FAST_DEPLOYMENT_URL=<full fast deployment URL>\n')
    for name, url in {'AZURE_OPENAI_GPT52_DEPLOYMENT_URL': AZURE_OPENAI_GPT52_DEPLOYMENT_URL, 'AZURE_OPENAI_FAST_DEPLOYMENT_URL': AZURE_OPENAI_FAST_DEPLOYMENT_URL}.items():
        if not str(url).startswith('https://'):
            raise ValueError(f'{name} must be a full HTTPS deployment URL: {url!r}')
        if '/chat/completions' not in str(url):
            raise ValueError(f'{name} does not look like a chat-completions URL.\nConfigured URL shape: {_mask_url_for_display(url)}\n\nExpected a full URL ending with /chat/completions plus any required query string.')
validate_llm_config()
print('Azure/OpenAI full-URL configuration loaded')
print('Strong endpoint:', _mask_url_for_display(AZURE_OPENAI_GPT52_DEPLOYMENT_URL))
print('Fast endpoint:', _mask_url_for_display(AZURE_OPENAI_FAST_DEPLOYMENT_URL))
print('Model routing:', json.dumps(MODEL_CONFIG, indent=2))

def get_model_url(model_tier: str='strong') -> str:
    model_tier = (model_tier or 'strong').lower().strip()
    if model_tier == 'fast':
        return AZURE_OPENAI_FAST_DEPLOYMENT_URL
    return AZURE_OPENAI_GPT52_DEPLOYMENT_URL

def _extract_message_content(data: Dict[str, Any]) -> str:
    try:
        content = data['choices'][0]['message']['content']
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError('Unexpected Azure/OpenAI response structure:\n' + json.dumps(data, indent=2, ensure_ascii=False)[:3000]) from exc
    if not content:
        raise ValueError('Azure/OpenAI returned an empty message content. Response:\n' + json.dumps(data, indent=2, ensure_ascii=False)[:3000])
    return content

def _azure_chat_completion(*, url: str, api_key: str, messages: List[Dict[str, str]], max_output_tokens: int, json_mode: bool=False, temperature: Optional[float]=None, timeout: int=240, request_label: str='LLM', max_attempts: int=6) -> Dict[str, Any]:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Mirrors the working logic you provided:
    - Uses full deployment URL directly.
    - Retries transient 500/502/503/504 and connection errors.
    - Handles 429 Retry-After.
    - Tries max_completion_tokens first, then max_tokens for gateway compatibility.
    - Does not expose API keys in errors.
    """
    token_fields = ['max_completion_tokens', 'max_tokens']
    last_error = None
    for token_field in token_fields:
        payload = {'messages': messages, token_field: max_output_tokens}
        if temperature is not None:
            payload['temperature'] = temperature
        if json_mode:
            payload['response_format'] = {'type': 'json_object'}
        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(url, data=json.dumps(payload).encode('utf-8'), headers={'Content-Type': 'application/json', 'api-key': api_key}, method='POST')
            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode('utf-8'))
            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors='replace')
                last_error = RuntimeError(f'{request_label} HTTP error {exc.code}.\nEndpoint: {_mask_url_for_display(url)}\nToken field used: {token_field}\nResponse: {body[:3000]}')
                rate_limited = exc.code == 429
                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = token_field == 'max_completion_tokens' and exc.code in {400, 422, 500}
                if rate_limited and attempt < max_attempts:
                    retry_after = None
                    try:
                        ra = exc.headers.get('Retry-After') if exc.headers else None
                        if ra is not None:
                            retry_after = float(str(ra).strip())
                    except (TypeError, ValueError):
                        retry_after = None
                    wait = retry_after if retry_after is not None else 2 ** attempt * 2 + random.random()
                    wait = min(wait, 90)
                    print(f'{request_label}: rate limited (429); retrying attempt {attempt + 1}/{max_attempts} in {wait:.1f}s...')
                    time.sleep(wait)
                    continue
                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(f'{request_label}: server error {exc.code}; retrying attempt {attempt + 1}/{max_attempts} in {wait:.1f}s...')
                    time.sleep(wait)
                    continue
                if compatibility_candidate:
                    print(f'{request_label}: gateway may not support `max_completion_tokens`; retrying with `max_tokens`.')
                    break
                if exc.code == 404:
                    raise RuntimeError(f'{request_label} HTTP 404 Resource not found.\nEndpoint: {_mask_url_for_display(url)}\n\nThe notebook is now sending the configured full URL directly. So a 404 means the URL itself is not accepted by the gateway, or the deployment behind that URL is not accessible with this key.\n\nCompare the exact .env value of AZURE_OPENAI_GPT52_DEPLOYMENT_URL with the endpoint URL that works in your other notebook.') from exc
                raise last_error from exc
            except urllib.error.URLError as exc:
                last_error = RuntimeError(f'{request_label} connection error.\nEndpoint: {_mask_url_for_display(url)}\nError: {exc}')
                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(f'{request_label}: connection issue; retrying attempt {attempt + 1}/{max_attempts} in {wait:.1f}s...')
                    time.sleep(wait)
                    continue
                raise last_error from exc
            except (ConnectionError, TimeoutError, OSError, http.client.RemoteDisconnected) as exc:
                last_error = RuntimeError(f'{request_label} connection reset/timeout.\nEndpoint: {_mask_url_for_display(url)}\nError: {type(exc).__name__}: {exc}')
                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(f'{request_label}: connection reset/timeout ({type(exc).__name__}); retrying attempt {attempt + 1}/{max_attempts} in {wait:.1f}s...')
                    time.sleep(wait)
                    continue
                raise last_error from exc
    raise last_error or RuntimeError(f'{request_label} request failed for an unknown reason.')

def _strip_markdown_json_fence(text: str) -> str:
    """Remove common ```json fences without touching the JSON body."""
    text = str(text or '').strip()
    if text.startswith('```'):
        text = re.sub('^```(?:json)?\\s*', '', text, flags=re.IGNORECASE)
        text = re.sub('\\s*```$', '', text)
    return text.strip()

def _extract_balanced_json_object(text: str) -> Optional[str]:
    """
    Return the first balanced JSON object found in text.

    This is safer than taking text[first_brace:last_brace] because model output can
    contain explanatory text, braces inside strings, or multiple JSON-looking blocks.
    If the object is truncated and never balances, return None so the caller can
    attempt LLM repair on the best candidate.
    """
    start = None
    depth = 0
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if start is None:
            if ch == '{':
                start = i
                depth = 1
            continue
        if escape:
            escape = False
            continue
        if ch == '\\':
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None

def _extract_json_object(text: str) -> str:
    """
    Extract the most likely JSON object from model output.

    Handles markdown fences and leading/trailing commentary. If the output appears
    truncated, returns the partial object candidate so the repair step can fix it.
    """
    text = _strip_markdown_json_fence(text)
    if text.startswith('{') and text.endswith('}'):
        return text
    balanced = _extract_balanced_json_object(text)
    if balanced:
        return balanced
    first = text.find('{')
    last = text.rfind('}')
    if first >= 0 and last > first:
        return text[first:last + 1]
    if first >= 0:
        return text[first:]
    return text

def _json_error_context(candidate: str, exc: json.JSONDecodeError, radius: int=300) -> str:
    """Small excerpt around a JSONDecodeError location for debugging."""
    pos = getattr(exc, 'pos', 0)
    left = max(0, pos - radius)
    right = min(len(candidate), pos + radius)
    excerpt = candidate[left:right]
    pointer = ' ' * max(0, pos - left) + '^'
    return excerpt + '\n' + pointer

def _safe_debug_filename(label: str) -> str:
    label = re.sub('[^A-Za-z0-9_.-]+', '_', str(label or 'llm_json'))
    return label.strip('_')[:80] or 'llm_json'

def _write_llm_json_debug(raw: str, label: str='malformed_json') -> Optional[Path]:
    """
    Persist malformed raw LLM output for inspection.
    Uses the notebook audit_logs folder when available.
    """
    try:
        base = DIRS.get('audit_logs', OUTPUT_DIR) if 'DIRS' in globals() else Path.cwd()
        base = Path(base) / 'llm_json_debug'
        base.mkdir(parents=True, exist_ok=True)
        path = base / f"{time.strftime('%Y%m%d_%H%M%S')}_{_safe_debug_filename(label)}_{uuid.uuid4().hex[:8]}.txt"
        path.write_text(str(raw), encoding='utf-8')
        return path
    except Exception:
        return None

def _repair_json_with_strong_model(malformed_content: str, request_label: str) -> Dict[str, Any]:
    repair_system = 'You repair malformed or truncated JSON. Return one complete valid JSON object only. Preserve the original meaning, scores, checklist values, issues, and fixes. Keep strings concise. Do not add markdown fences or commentary.'
    repair_user = f'\nRepair the following malformed or truncated output into one complete valid JSON object.\n\nRequirements:\n- Keep the same top-level fields when present.\n- Finish incomplete strings and arrays conservatively.\n- Fix missing commas, unescaped quotes, dangling keys, and truncated arrays.\n- Limit each issue/fix/support note string to at most 35 words.\n- If the object is a claims register, preserve as many claims as possible but cap at 60 claims.\n- Return JSON only.\n\nMALFORMED OUTPUT:\n{str(malformed_content)[:70000]}\n'.strip()
    data = _azure_chat_completion(url=AZURE_OPENAI_GPT52_DEPLOYMENT_URL, api_key=AZURE_OPENAI_API_KEY, messages=[{'role': 'system', 'content': repair_system}, {'role': 'user', 'content': repair_user}], max_output_tokens=int(os.getenv('JSON_REPAIR_MAX_TOKENS', '6000')), json_mode=True, temperature=0, request_label=f'{request_label} JSON repair')
    repaired = _extract_message_content(data)
    candidate = _extract_json_object(repaired)
    return json.loads(candidate)

def _parse_or_repair_json(raw: str, request_label: str='LLM output') -> Dict[str, Any]:
    """
    Parse JSON returned by an LLM. If parsing fails, save the raw output and ask
    the strong model to repair it. This prevents one malformed JSON response from
    crashing the full generation pipeline.
    """
    candidate = _extract_json_object(raw)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError as exc:
        debug_path = _write_llm_json_debug(raw, request_label)
        print(f'{request_label}: invalid JSON at line {exc.lineno}, column {exc.colno}. Attempting JSON repair...')
        if debug_path:
            print('Raw malformed output saved to:', debug_path)
        try:
            return _repair_json_with_strong_model(candidate, request_label=request_label)
        except Exception as repair_exc:
            detail = _json_error_context(candidate, exc)
            raise ValueError(f'{request_label}: failed to parse JSON and repair also failed.\nOriginal JSON error: {exc}\nDebug file: {debug_path}\nError context:\n{detail}') from repair_exc

def azure_chat(messages: List[Dict[str, str]], model_tier: str='strong', temperature: float=0, max_tokens: int=4000, response_format: Optional[Dict[str, str]]=None, retries: int=6, retry_sleep: int=3) -> str:
    """
    Azure/OpenAI Chat Completions helper used by all LLM agents.

    Returns text content.
    If response_format={"type": "json_object"}, the model is asked for JSON mode.
    """
    url = get_model_url(model_tier)
    request_label = f'Azure {model_tier} agent'
    json_mode = bool(response_format and response_format.get('type') == 'json_object')
    data = _azure_chat_completion(url=url, api_key=AZURE_OPENAI_API_KEY, messages=messages, max_output_tokens=max_tokens, json_mode=json_mode, temperature=temperature, request_label=request_label, max_attempts=retries)
    return _extract_message_content(data)

def azure_chat_json(messages: List[Dict[str, str]], model_tier: str='strong', temperature: float=0, max_tokens: int=4000, retries: int=6, request_label: Optional[str]=None) -> Dict[str, Any]:
    """
    JSON-safe LLM call.
    First requests JSON mode. If the model returns malformed or truncated JSON,
    parse_json_response repairs it with the strong model.
    """
    label = request_label or f'Azure {model_tier} agent'
    content = azure_chat(messages=messages, model_tier=model_tier, temperature=temperature, max_tokens=max_tokens, response_format={'type': 'json_object'}, retries=retries)
    return parse_json_response(content, request_label=label)

def parse_json_response(raw: str, request_label: str='LLM output') -> Dict[str, Any]:
    """
    Backward-compatible parser for cells that call azure_chat(...json mode...).
    Now robust: strict parse first, then automatic repair instead of a hard crash.
    """
    return _parse_or_repair_json(raw, request_label=request_label)

def parse_json_safely(raw: str) -> Dict[str, Any]:
    try:
        return parse_json_response(raw)
    except Exception:
        return {'parse_error': True, 'raw_output_preview': str(raw)[:2000]}

def test_llm_connection(model_tier: str='strong') -> None:
    """Quick smoke test for a configured endpoint."""
    print(f'Testing {model_tier} endpoint:', _mask_url_for_display(get_model_url(model_tier)))
    content = azure_chat([{'role': 'user', 'content': 'Return exactly: OK'}], model_tier=model_tier, temperature=0, max_tokens=20)
    print(f'{model_tier} response:', content)
print('Full-URL LLM helper functions ready')
print("Run test_llm_connection('strong') and test_llm_connection('fast') before running the full pipeline.")

## Clean engine

In [ ]:
import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Protocol, TypedDict

from langgraph.graph import StateGraph, END

### Settings

In [ ]:
@dataclass(frozen=True)
class Thresholds:
    judge_min: float = 7.0           # each judge must score >= this
    max_revisions: int = 3           # revision attempts before human review
    min_section_words: int = 120     # structural floor


@dataclass(frozen=True)
class Settings:
    dirs: Dict[str, Path]                 # the old notebook's DIRS
    sections: List[str]                   # the old notebook's SECTIONS
    section_slugs: Dict[str, str]         # the old notebook's SECTION_SLUGS
    thresholds: Thresholds = field(default_factory=Thresholds)
    writer_max_steps: int = 6
    reviser_max_steps: int = 6
    model_tier: str = "strong"
    max_tokens: int = 4000

### Domain models (typed)

In [ ]:
@dataclass(frozen=True)
class EvidenceItem:
    id: str
    text: str
    value: Optional[str] = None

    def haystack(self) -> str:
        return f"{self.text} {self.value or ''}".lower()


@dataclass(frozen=True)
class Requirement:
    id: str
    text: str
    coverage: str = "unknown"        # covered | partial | not_available


@dataclass(frozen=True)
class SectionContext:
    """Immutable inputs for one section, loaded from prep artifacts."""
    section_name: str
    slug: str
    requirements: List[Requirement]
    evidence: List[EvidenceItem]
    disclosure_plan: Dict[str, Any]

    def supported_requirements(self) -> List[Requirement]:
        return [r for r in self.requirements if r.coverage in ("covered", "partial")]


@dataclass(frozen=True)
class Claim:
    text: str
    value: str
    evidence_id: Optional[str]       # None => unsupported

    @property
    def supported(self) -> bool:
        return self.evidence_id is not None


@dataclass(frozen=True)
class GateFinding:
    gate: str
    passed: bool
    issues: List[str] = field(default_factory=list)


@dataclass
class GateReport:
    findings: List[GateFinding]

    @property
    def passed(self) -> bool:
        return all(f.passed for f in self.findings)

    def failures(self) -> List[Dict[str, Any]]:
        return [{"gate": f.gate, "issues": f.issues} for f in self.findings if not f.passed]


@dataclass(frozen=True)
class JudgeScore:
    kind: str
    score: float
    issues: List[str] = field(default_factory=list)


@dataclass
class JudgeReport:
    scores: List[JudgeScore]

    def min_score(self) -> float:
        return min((s.score for s in self.scores), default=0.0)

    def below(self, threshold: float) -> List[JudgeScore]:
        return [s for s in self.scores if s.score < threshold]


@dataclass
class Approval:
    approved: bool
    failures: List[Dict[str, Any]]
    overall_score: float

### IO — load your prep artifacts

In [ ]:
class ContextLoader:
    """Loads prep artifacts produced by the existing prep phase into typed SectionContext.

    Field names are defensive: prep JSON is read leniently so schema drift degrades to
    empty lists rather than crashes. Adjust the small `_extract_*` helpers if your prep
    uses different keys."""

    def __init__(self, settings: Settings):
        self.s = settings

    def _read(self, path: Path) -> Any:
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

    def load(self, section_name: str) -> SectionContext:
        slug = self.s.section_slugs[section_name]
        coverage = self._read(self.s.dirs["coverage"] / f"coverage_matrix_{slug}.json") or {}
        evmap = self._read(self.s.dirs["evidence_maps"] / f"evidence_map_{slug}.json") or {}
        plan = self._read(self.s.dirs["plans"] / f"disclosure_plan_{slug}.json") or {}
        return SectionContext(
            section_name=section_name, slug=slug,
            requirements=self._extract_requirements(coverage),
            evidence=self._extract_evidence(evmap),
            disclosure_plan=plan if isinstance(plan, dict) else {"plan": plan},
        )

    @staticmethod
    def _extract_requirements(coverage: Any) -> List[Requirement]:
        rows = coverage if isinstance(coverage, list) else (
            coverage.get("requirements", []) if isinstance(coverage, dict) else [])
        cov_map = {"covered": "covered", "partially_covered": "partial", "partial": "partial"}
        out: List[Requirement] = []
        for r in (rows or []):
            if not isinstance(r, dict):
                continue
            status = str(r.get("coverage_status", r.get("coverage", ""))).lower()
            out.append(Requirement(
                id=str(r.get("requirement_id", r.get("id", ""))),
                text=str(r.get("requirement_text", r.get("text", ""))),
                coverage=cov_map.get(status, "not_available")))
        return out

    @staticmethod
    def _extract_evidence(evmap: Any) -> List[EvidenceItem]:
        """Flatten writer-safe evidence_candidates across all requirements. Audit-only and
        missing-like candidates are excluded so they never reach the writer."""
        rows = evmap if isinstance(evmap, list) else (
            evmap.get("evidence", evmap.get("items", [])) if isinstance(evmap, dict) else [])
        seen, out = set(), []
        for req in (rows or []):
            if not isinstance(req, dict):
                continue
            for c in req.get("evidence_candidates", []):
                if not isinstance(c, dict):
                    continue
                if not c.get("writer_safe", True) or c.get("audit_only_evidence") or c.get("missing_like_value"):
                    continue
                eid = str(c.get("payload_path", c.get("payload_root", "")))
                val = c.get("value_preview")
                key = (eid, str(val))
                if not eid or key in seen:
                    continue
                seen.add(key)
                kws = ", ".join(str(k) for k in c.get("matched_keywords", [])[:6])
                out.append(EvidenceItem(id=eid, text=(f"{eid} — {kws}" if kws else eid),
                                        value=(str(val) if val is not None else None)))
        return out

### LLM — AzureChatModel over your azure_chat

In [ ]:
@dataclass
class ToolCall:
    name: str
    args: Dict[str, Any]
    id: str = ""


@dataclass
class AIResponse:
    content: str = ""
    tool_calls: List[ToolCall] = field(default_factory=list)


class ChatModel(Protocol):
    def complete(self, messages: List[Dict[str, Any]], tools: Optional[List["Tool"]] = None) -> AIResponse: ...


class AzureChatModel:
    """Wraps the old notebook's `azure_chat(messages, model_tier, ...) -> str`.

    That client returns text (no native tool calling), so tools are driven by a small JSON
    action protocol: the model is asked to reply with either
        {"action": "tool", "tool": "<name>", "args": {...}}   or   {"action": "final", "final": "..."}
    Plain (non-JSON) replies are treated as the final answer, so the writer still works even if
    it ignores the protocol. Judge calls (no tools) use JSON mode when the prompt asks for JSON."""

    def __init__(self, azure_chat: Callable, model_tier: str = "strong", max_tokens: int = 4000):
        self._chat = azure_chat
        self._tier = model_tier
        self._max = max_tokens

    @staticmethod
    def _to_plain(messages: List[Dict[str, Any]]) -> List[Dict[str, str]]:
        """Flatten agent messages (assistant tool_calls / role 'tool') into plain chat roles."""
        out = []
        for m in messages:
            role = m["role"]
            if role == "tool":
                out.append({"role": "user", "content": f"TOOL RESULT {m.get('content', '')}"})
            elif role == "assistant" and m.get("tool_calls"):
                out.append({"role": "assistant", "content": m.get("content") or "(calling tool)"})
            else:
                out.append({"role": role, "content": m.get("content", "")})
        return out

    def complete(self, messages, tools=None):
        plain = self._to_plain(messages)
        if not tools:
            wants_json = any("json" in m.get("content", "").lower()
                             for m in plain if m["role"] == "system")
            fmt = {"type": "json_object"} if wants_json else None
            content = self._chat(plain, model_tier=self._tier, max_tokens=self._max,
                                 temperature=0, response_format=fmt)
            return AIResponse(content=content)

        spec = "\n".join(f"- {t.name}({', '.join(t.func.__code__.co_varnames[:t.func.__code__.co_argcount])}): {t.description}"
                         for t in tools)
        protocol = ("You may use tools. Reply with ONE JSON object and nothing else: either "
                    '{"action":"tool","tool":"<name>","args":{...}} to call a tool, or '
                    '{"action":"final","final":"<your Markdown answer>"} when done.\n'
                    f"Available tools:\n{spec}")
        content = self._chat(plain + [{"role": "system", "content": protocol}],
                             model_tier=self._tier, max_tokens=self._max, temperature=0,
                             response_format={"type": "json_object"})
        action = parse_json_object(content)
        if action and action.get("action") == "tool" and action.get("tool"):
            return AIResponse(tool_calls=[ToolCall(action["tool"], action.get("args") or {}, action["tool"])])
        if action and action.get("action") == "final":
            return AIResponse(content=action.get("final", ""))
        return AIResponse(content=content)   # non-JSON -> treat as final answer


def parse_json_object(text: str) -> Optional[dict]:
    try:
        return json.loads(text[text.index("{"): text.rindex("}") + 1])
    except Exception:
        return None

### Tools + bounded agent loop

In [ ]:
@dataclass
class Tool:
    name: str
    description: str
    func: Callable[..., Any]


def retrieval_tools(ctx: SectionContext) -> List[Tool]:
    return [
        Tool("get_requirements", "IFRS requirements assigned to this section (id, text, coverage).",
             lambda: json.dumps([r.__dict__ for r in ctx.supported_requirements()])[:6000]),
        Tool("get_evidence", "BANK01 evidence available for this section (id, text, value).",
             lambda: json.dumps([e.__dict__ for e in ctx.evidence])[:9000]),
        Tool("get_disclosure_plan", "The disclosure plan/blueprint for this section.",
             lambda: json.dumps(ctx.disclosure_plan)[:6000]),
        Tool("search_evidence", "Search this section's evidence for a keyword or figure.",
             lambda query: json.dumps([e.__dict__ for e in ctx.evidence
                                       if str(query).lower() in e.haystack()][:20])[:9000]),
    ]


class ToolAgent:
    """Bounded, fully-logged tool loop. Returns (final_text, trace)."""
    def __init__(self, model: ChatModel, tools: List[Tool], max_steps: int):
        self.model, self.tools = model, tools
        self.reg = {t.name: t for t in tools}
        self.max_steps = max_steps

    def run(self, system: str, user: str):
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        trace: List[Dict[str, Any]] = []
        for _ in range(self.max_steps):
            resp = self.model.complete(messages, tools=self.tools)
            if not resp.tool_calls:
                return resp.content, trace
            messages.append({"role": "assistant", "content": resp.content,
                             "tool_calls": [tc.__dict__ for tc in resp.tool_calls]})
            for tc in resp.tool_calls:
                try:
                    obs = self.reg[tc.name].func(**tc.args)
                except Exception as e:
                    obs = f"ERROR: {e}"
                trace.append({"tool": tc.name, "args": tc.args})
                messages.append({"role": "tool", "content": f"[{tc.name}] {obs}"[:12000], "tool_call_id": tc.id})
        final = self.model.complete(messages + [{"role": "user", "content": "Provide the final Markdown now."}])
        return final.content, trace

### Claims (deterministic)

In [ ]:
_NUMERIC = re.compile(
    r"(?<![\w.])(?:[€$£]\s?)?\d+(?:,\d{3})*(?:\.\d+)?"
    r"(?:\s?(?:tCO2e|tCO\u2082e|kt|MWh|GWh|EUR|USD|years?|%|t)\b)?", re.I)


def build_claims(draft: str, ctx: SectionContext) -> List[Claim]:
    """Extract quantitative claims (numbers/units/years) and map each to an evidence item
    by value overlap. A claim with no matching evidence is 'unsupported' -> fails grounding."""
    claims: List[Claim] = []
    for m in _NUMERIC.finditer(draft):
        value = m.group(0).strip().rstrip(",.;:")
        if len(re.sub(r"[^\d]", "", value)) < 2:   # skip trivial single digits
            continue
        norm = re.sub(r"[,\s]", "", value).lower()
        ev = next((e for e in ctx.evidence if norm and norm in re.sub(r"[,\s]", "", e.haystack())), None)
        claims.append(Claim(text=draft[max(0, m.start() - 40):m.end() + 20].strip(),
                            value=value, evidence_id=ev.id if ev else None))
    return claims

### Deterministic gates

In [ ]:
_PLACEHOLDERS = re.compile(r"\[[^\]]*(insert|tbd|add|placeholder|xxx)[^\]]*\]|\bTBD\b|\bXXX\b", re.I)
_MISSING_LANG = re.compile(
    r"\b(not\s+(?:available|reported|disclosed|provided)|no\s+data|data\s+gap|unavailable|"
    r"not\s+applicable\s+data|absence\s+of\s+data)\b", re.I)


def grounding_gate(claims: List[Claim]) -> GateFinding:
    unsupported = [c.value for c in claims if not c.supported]
    return GateFinding("grounding", not unsupported,
                       [f"claim not traceable to evidence: {v}" for v in unsupported[:10]])


def cleanliness_gate(draft: str) -> GateFinding:
    issues = []
    if _PLACEHOLDERS.search(draft):
        issues.append("contains template placeholders")
    if _MISSING_LANG.search(draft):
        issues.append("contains missing/unavailable-data language")
    return GateFinding("cleanliness", not issues, issues)


def structure_gate(draft: str, ctx: SectionContext, min_words: int) -> GateFinding:
    issues = []
    if len(draft.split()) < min_words:
        issues.append(f"below minimum length ({min_words} words)")
    if not re.search(r"^#{1,3}\s", draft, re.M):
        issues.append("no Markdown headings")
    return GateFinding("structure", not issues, issues)


def claims_integrity_gate(claims: List[Claim]) -> GateFinding:
    # every extracted claim must have a value; supported ones must reference evidence
    issues = [f"malformed claim: {c.text[:40]}" for c in claims if not c.value]
    return GateFinding("claims_integrity", not issues, issues)


def run_gates(draft: str, ctx: SectionContext, settings: Settings) -> tuple[GateReport, List[Claim]]:
    claims = build_claims(draft, ctx)
    report = GateReport([
        grounding_gate(claims),
        cleanliness_gate(draft),
        structure_gate(draft, ctx, settings.thresholds.min_section_words),
        claims_integrity_gate(claims),
    ])
    return report, claims

### Structured judges

In [ ]:
_JUDGE_PROMPTS = {
    "coverage": ("Score how completely the section addresses its assigned IFRS requirements. "
                 "Penalise omitted mandatory disclosures."),
    "evidence": ("Score how well every quantitative and factual claim is supported by the provided "
                 "evidence. Penalise any claim not traceable to evidence."),
    "style": ("Score IFRS-appropriate tone, clarity and neutrality. Penalise marketing language, "
              "hedging, placeholders, or references to missing data."),
}


def run_judges(model: ChatModel, draft: str, ctx: SectionContext) -> JudgeReport:
    evidence = json.dumps([e.__dict__ for e in ctx.evidence])[:6000]
    reqs = json.dumps([r.__dict__ for r in ctx.supported_requirements()])[:4000]
    scores: List[JudgeScore] = []
    for kind, rubric in _JUDGE_PROMPTS.items():
        system = (f"You are an IFRS S1/S2 {kind} judge. {rubric} "
                  f'Return ONLY JSON: {{"score_0_to_10": <number>, "issues": [<strings>]}}.')
        user = f"REQUIREMENTS:\n{reqs}\n\nEVIDENCE:\n{evidence}\n\nSECTION:\n{draft}"
        parsed = parse_json_object(model.complete([{"role": "system", "content": system},
                                                   {"role": "user", "content": user}]).content) or {}
        scores.append(JudgeScore(kind=kind, score=float(parsed.get("score_0_to_10", 0.0)),
                                 issues=list(parsed.get("issues", []))))
    return JudgeReport(scores)

### Approval (single source of truth)

In [ ]:
def decide_approval(gates: GateReport, judges: Optional[JudgeReport], settings: Settings) -> Approval:
    failures = list(gates.failures())
    judge_min = settings.thresholds.judge_min
    if judges is not None:
        for js in judges.below(judge_min):
            failures.append({"gate": f"judge_{js.kind}", "issues": js.issues or [f"score {js.score} < {judge_min}"]})
    overall = judges.min_score() if judges is not None else 0.0
    approved = gates.passed and judges is not None and not judges.below(judge_min)
    return Approval(approved=approved, failures=failures, overall_score=overall)

### Writer + reviser agents

In [ ]:
WRITER_RULES = (
    "Write final-report IFRS prose grounded strictly in retrieved evidence. Use real values from "
    "get_evidence; never use placeholders, template rows, or instructions to the entity. Never write "
    "about missing/unavailable data; if evidence is partial, write only the supported subset. Do not "
    "invent committees, policies, targets, metrics, dates, currencies or figures. Every quantitative "
    "claim must be traceable to an evidence item. Use neutral IFRS-aligned language.")

WRITER_SYSTEM = ("You are an IFRS S1/S2 disclosure writer. First call get_requirements, get_evidence and "
                 "get_disclosure_plan; use search_evidence for specific figures. Then write the section in "
                 "Markdown with appropriate headings. " + WRITER_RULES)
REVISER_SYSTEM = ("You are an IFRS reviser. Read the listed failures, fetch evidence with the tools, and make "
                  "the minimal corrections needed. Return the corrected Markdown only. " + WRITER_RULES)


def write_draft(model: ChatModel, ctx: SectionContext, settings: Settings) -> tuple[str, list]:
    agent = ToolAgent(model, retrieval_tools(ctx), settings.writer_max_steps)
    return agent.run(WRITER_SYSTEM, f"Write the IFRS section '{ctx.section_name}'.")


def revise_draft(model: ChatModel, ctx: SectionContext, draft: str,
                 failures: List[Dict[str, Any]], settings: Settings) -> tuple[str, list]:
    agent = ToolAgent(model, retrieval_tools(ctx), settings.reviser_max_steps)
    user = f"Section '{ctx.section_name}' failed: {failures}\n\nCurrent draft:\n\n{draft}\n\nReturn corrected Markdown."
    return agent.run(REVISER_SYSTEM, user)

### LangGraph loop

In [ ]:
class SectionState(TypedDict, total=False):
    ctx: SectionContext
    draft: str
    claims: List[Claim]
    gates: GateReport
    judges: Optional[JudgeReport]
    approval: Approval
    iteration: int
    prev_failures: Any
    trace: List[Dict[str, Any]]
    result: Dict[str, Any]


def build_graph(model: ChatModel, settings: Settings):

    def n_generate(s: SectionState) -> SectionState:
        draft, trace = write_draft(model, s["ctx"], settings)
        return {"draft": draft, "iteration": 0, "prev_failures": None,
                "trace": [{"agent": "writer", "steps": trace}]}

    def n_evaluate(s: SectionState) -> SectionState:
        gates, claims = run_gates(s["draft"], s["ctx"], settings)
        return {"gates": gates, "claims": claims}

    def route_gates(s: SectionState) -> str:
        return "judge" if s["gates"].passed else "approve"

    def n_judge(s: SectionState) -> SectionState:
        return {"judges": run_judges(model, s["draft"], s["ctx"])}

    def n_approve(s: SectionState) -> SectionState:
        approval = decide_approval(s["gates"], s.get("judges"), settings)
        return {"approval": approval}

    def route_approve(s: SectionState) -> str:
        if s["approval"].approved:
            return "approved"
        cur = json.dumps(s["approval"].failures, sort_keys=True, default=str)
        if cur == s["prev_failures"] and s["gates"].passed:
            return "human_review"          # same issue persists -> escalate
        if s["iteration"] >= settings.thresholds.max_revisions:
            return "human_review"
        return "revise"

    def n_revise(s: SectionState) -> SectionState:
        draft, trace = revise_draft(model, s["ctx"], s["draft"], s["approval"].failures, settings)
        prev = json.dumps(s["approval"].failures, sort_keys=True, default=str)
        return {"draft": draft, "iteration": s["iteration"] + 1, "prev_failures": prev,
                "trace": [{"agent": "reviser", "iteration": s["iteration"], "steps": trace}]}

    def n_approved(s: SectionState) -> SectionState:
        return {"result": _finish(s, "approved", settings)}

    def n_human(s: SectionState) -> SectionState:
        return {"result": _finish(s, "human_review", settings)}

    g = StateGraph(SectionState)
    for name, fn in [("generate", n_generate), ("evaluate", n_evaluate), ("judge", n_judge),
                     ("approve", n_approve), ("revise", n_revise),
                     ("approved", n_approved), ("human_review", n_human)]:
        g.add_node(name, fn)
    g.set_entry_point("generate")
    g.add_edge("generate", "evaluate")
    g.add_conditional_edges("evaluate", route_gates, {"judge": "judge", "approve": "approve"})
    g.add_edge("judge", "approve")
    g.add_conditional_edges("approve", route_approve,
                            {"approved": "approved", "revise": "revise", "human_review": "human_review"})
    g.add_edge("revise", "evaluate")
    g.add_edge("approved", END)
    g.add_edge("human_review", END)
    return g.compile()


def _finish(s: SectionState, status: str, settings: Settings) -> Dict[str, Any]:
    ctx = s["ctx"]
    record = {
        "section_name": ctx.section_name, "slug": ctx.slug, "status": status,
        "iterations": s.get("iteration", 0), "markdown": s["draft"],
        "overall_score": s["approval"].overall_score if s.get("approval") else 0.0,
        "failures": s["approval"].failures if s.get("approval") else [],
        "claims": [c.__dict__ for c in s.get("claims", [])],
        "agent_trace": s.get("trace", []),
    }
    prefix = "approved" if status == "approved" else "human_review"
    _write_json(record, settings.dirs["approved"] / f"{prefix}_{ctx.slug}.json")
    _write_text(s["draft"], settings.dirs["approved"] / f"{prefix}_{ctx.slug}.md")
    return record

### Assembly + run

In [ ]:
def _write_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def run_section(section_name: str, loader: ContextLoader, model: ChatModel, settings: Settings) -> Dict[str, Any]:
    graph = build_graph(model, settings)
    limit = (settings.thresholds.max_revisions + 1) * 6 + 10
    out = graph.invoke({"ctx": loader.load(section_name)}, config={"recursion_limit": limit})
    return out["result"]


def run_report(loader: ContextLoader, model: ChatModel, settings: Settings,
               sections: Optional[List[str]] = None) -> Dict[str, Any]:
    results = [run_section(sn, loader, model, settings) for sn in (sections or settings.sections)]
    report_md = assemble_report(results)
    _write_text(report_md, settings.dirs["handoff"] / "approved_report.md")
    _write_json({"sections": results, "approved": sum(r["status"] == "approved" for r in results),
                 "total": len(results)}, settings.dirs["audit_logs"] / "run_summary.json")
    return {"results": results, "report_markdown": report_md}


def assemble_report(results: List[Dict[str, Any]]) -> str:
    parts = ["# IFRS S1/S2 Sustainability Report\n"]
    for r in results:
        flag = "" if r["status"] == "approved" else "  _(pending human review)_"
        parts.append(f"\n## {r['section_name']}{flag}\n\n{r['markdown'].strip()}\n")
    return "\n".join(parts)

## Self-contained smoke test (no Azure, no data)

In [ ]:
def _smoke_test():
    import json, tempfile
    from pathlib import Path
    tmp = Path(tempfile.mkdtemp())
    dirs = {k: tmp / k for k in ("evidence_maps", "coverage", "plans", "approved", "handoff", "audit_logs", "revisions")}
    for p in dirs.values(): p.mkdir(parents=True, exist_ok=True)
    st = Settings(dirs=dirs, sections=["Governance"], section_slugs={"Governance": "governance"},
                  thresholds=Thresholds(min_section_words=90))

    # fixtures in the REAL prep schema
    json.dump([{"requirement_id": "G1", "requirement_text": "Board oversight of climate risks",
                "coverage_status": "covered", "mandatory": True}],
              open(dirs["coverage"] / "coverage_matrix_governance.json", "w"))
    json.dump([{"requirement_id": "G1", "requirement_text": "Board oversight of climate risks", "mandatory": True,
                "evidence_candidates": [
                    {"payload_path": "governance[0].reporting_year", "value_preview": "2024",
                     "writer_safe": True, "audit_only_evidence": False, "missing_like_value": False,
                     "matched_keywords": ["year", "governance"]},
                    {"payload_path": "metrics[0].emissions_reduction", "value_preview": "1250 tCO2e",
                     "writer_safe": True, "audit_only_evidence": False, "missing_like_value": False,
                     "matched_keywords": ["emissions"]}]}],
              open(dirs["evidence_maps"] / "evidence_map_governance.json", "w"))
    json.dump({"subsections": ["Board oversight"]}, open(dirs["plans"] / "disclosure_plan_governance.json", "w"))

    GOOD = """## Board oversight

The board maintains oversight of climate-related risks through its sustainability committee. During the period ending 2024
the committee reviewed the entity's climate strategy, transition plan and progress against approved targets, and considered
management's assessment of climate risks across the organisation. The committee integrates climate considerations into the
overall risk governance framework and reports to the full board regularly.

## Management role

Management monitors climate metrics, oversees implementation of climate policies and tracks operational performance,
including a reduction of 1250 tCO2e over the period, reporting to the board committee on delivery of climate commitments."""

    class FakeModel:
        def complete(self, messages, tools=None):
            joined = " ".join(m.get("content", "") for m in messages).lower()
            if tools:
                return (AIResponse(tool_calls=[ToolCall("get_evidence", {}, "e")])
                        if "tool result" not in joined else AIResponse(content=GOOD))
            if "judge" in joined:
                return AIResponse(content='{"score_0_to_10": 8, "issues": []}')
            return AIResponse(content=GOOD)

    r = run_report(ContextLoader(st), FakeModel(), st)["results"][0]
    assert r["status"] == "approved" and all(c["evidence_id"] for c in r["claims"]), r
    print("positive: approved @ iter", r["iterations"], "| score", r["overall_score"],
          "| claims", [(c["value"], c["evidence_id"]) for c in r["claims"]])

    BAD = "## Oversight\n\nThe board met [INSERT NUMBER] times and cut emissions by 999 tCO2e. Data not available for targets."
    class BadModel(FakeModel):
        def complete(self, messages, tools=None):
            joined = " ".join(m.get("content", "") for m in messages).lower()
            if tools:
                return (AIResponse(tool_calls=[ToolCall("get_evidence", {}, "e")])
                        if "tool result" not in joined else AIResponse(content=BAD))
            if "judge" in joined: return AIResponse(content='{"score_0_to_10": 8}')
            return AIResponse(content=BAD)

    r2 = run_report(ContextLoader(st), BadModel(), st)["results"][0]
    assert r2["status"] == "human_review", r2
    print("negative: held for review | failing gates:", sorted({f["gate"] for f in r2["failures"]}))
    print("CLEAN ENGINE VERIFIED.")


_smoke_test()


## Production run (your Azure client + your DIRS)

In [ ]:
# Uses azure_chat, DIRS, SECTIONS, SECTION_SLUGS defined in the setup cells above.
model = AzureChatModel(azure_chat, model_tier="strong", max_tokens=4000)
settings = Settings(dirs=DIRS, sections=SECTIONS, section_slugs=SECTION_SLUGS)

result = run_report(ContextLoader(settings), model, settings)
print("approved:", sum(r["status"] == "approved" for r in result["results"]), "/", len(result["results"]))
print(result["report_markdown"][:2000])


### How to run
1. Run your **prep phase** once so `DIRS['evidence_maps']`, `DIRS['coverage']`, `DIRS['plans']` contain
   the per-section artifacts.
2. `pip install langgraph`, then run the two setup cells and every engine cell.
3. Run the **smoke test** (no Azure needed) to confirm graph/gates/judges/approval.
4. Ensure your `.env` has the Azure URLs the setup cell expects, then run the **production** cell.

Tuning: `Thresholds` (judge_min, max_revisions, min_section_words), the writer/judge prompts, and the
claim/gate regexes. The `ContextLoader._extract_*` helpers read your prep JSON leniently — adjust their
keys if your evidence/coverage schema differs.